In [ ]:
%pip install langchain_openai

In [8]:
from dotenv import load_dotenv, find_dotenv
from pathlib import Path

# Load closest .env (works whether running from repo root or dev/)
load_dotenv(find_dotenv(), override=False)


True

In [9]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
  model="gpt-4o",
  # reasoning_effort="low",
  # verbosity="low"
)

In [23]:
output_form = """
{
  "selected": "선택된 검색어",
  "real_keyword": "실제 검색어로 사용할 단어",
  "reason": "해당 검색어를 선택한 이유"
}
"""

system_prompt = """
당신은 트랜드 사이에서 검색어를 고민하는 쇼핑몰 유저입니다.
입력으로 현재 트랜드가 된 검색어들의 배열이 주어집니다.
해당 배열에서 하나의 요소를 뽑은 뒤, 그 요소와 관련된 검색어를 만들어 쇼핑몰에서 제품을 검색해야합니다.

markdown 문법을 사용하지 마십시오. 출력은 순수한 JSON으로 이루어져야 합니다.
검색어를 선택한 이유는 최대한 간단히 적으십시오.
검색어와 이유를 최대한 한국어로만 작성하십시오.

출력 양식은 다음과 같습니다:

"""


In [24]:
def build_prompt(prompt: str, form: str) -> str:
  return prompt + form

def filter_markdwon(text: str) -> str:
  return text.replace("```json", "").replace("```", "")

In [25]:
from langchain.schema import SystemMessage, HumanMessage, AIMessage

def call_llm(arrays: list[str]) -> AIMessage:
  texts = ""
  for item in arrays:
    texts = texts + ", " + item 
  messages = [
      SystemMessage(content=build_prompt(system_prompt, output_form)),
      HumanMessage(content=texts),
  ]

  result = llm.invoke(messages)

  # 결과값의 .content: 출력물
  return result.content.strip()

In [26]:
items = [
  "sports summary",
  "한서대",
  "고려대",
  "고려대학교",
  "을지대",
  "강민호",
  "엽기토끼 살인사건",
  "주우재",
  "miss universe 2025",
  "england vs australia"
]

In [27]:
output = call_llm(items)
output

'{\n  "selected": "miss universe 2025",\n  "real_keyword": "미스 유니버스 2025 드레스",\n  "reason": "미스 유니버스 대회는 패션과 관련된 콘텐츠가 많고, 드레스는 특히 관심을 많이 받는 아이템 중 하나입니다. 쇼핑몰에서 관련 드레스를 찾기에 적절한 검색어입니다."\n}'

In [28]:
print(output.strip())

{
  "selected": "miss universe 2025",
  "real_keyword": "미스 유니버스 2025 드레스",
  "reason": "미스 유니버스 대회는 패션과 관련된 콘텐츠가 많고, 드레스는 특히 관심을 많이 받는 아이템 중 하나입니다. 쇼핑몰에서 관련 드레스를 찾기에 적절한 검색어입니다."
}
